# RAG as a Python Pipeline
Answer from a fictional policy while making every retrieval step visible.

## 1. Load the policy

In [ ]:
from pathlib import Path

def load_policy(path="data/mini_policy.md"):
    return Path(path).read_text()

policy_text = load_policy()
print(policy_text[:180])

## 2. Split headed chunks

In [ ]:
def split_policy(text):
    sections = [part.strip() for part in text.split("# ") if part.strip()]
    return [{"id": f"P{i+1}", "text": section} for i, section in enumerate(sections)]

chunks = split_policy(policy_text)
print(chunks)

## 3. Create TF-IDF vectors

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer

def create_vectors(chunks):
    vectorizer = TfidfVectorizer()
    matrix = vectorizer.fit_transform(chunk["text"] for chunk in chunks)
    return vectorizer, matrix

vectorizer, matrix = create_vectors(chunks)
print(matrix.shape)

TF-IDF is a transparent teaching representation, not a modern semantic embedding model.

## 4. Retrieve chunks

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity

def retrieve_chunks(question, chunks, vectorizer, matrix, limit=2):
    query_vector = vectorizer.transform([question])
    scores = cosine_similarity(query_vector, matrix)[0]
    ranked = sorted(zip(chunks, scores), key=lambda item: item[1], reverse=True)
    return ranked[:limit]

question = "How much cash must remain?"
retrieved = retrieve_chunks(question, chunks, vectorizer, matrix)

## 5. Inspect IDs and scores

In [ ]:
for chunk, score in retrieved:
    print(chunk["id"], round(float(score), 3))

## 6. Build context

In [ ]:
def build_context(retrieved):
    return "\n\n".join(f"[{chunk['id']}] {chunk['text']}" for chunk, _ in retrieved)

context = build_context(retrieved)
print(context)

## 7. Build a grounded answer step

In [ ]:
import os

def generate_answer(question, context):
    prompt = f"Answer using only this context and cite IDs.\n{context}\nQuestion: {question}"
    configured = all(os.getenv(name) for name in ["MODEL_ENDPOINT", "MODEL_API_KEY", "MODEL_NAME"])
    if configured and os.getenv("COURSEWARE_OFFLINE", "0") != "1":
        from openai import OpenAI
        client = OpenAI(base_url=os.environ["MODEL_ENDPOINT"], api_key=os.environ["MODEL_API_KEY"])
        return client.chat.completions.create(model=os.environ["MODEL_NAME"], messages=[{"role": "user", "content": prompt}]).choices[0].message.content
    return "Offline example: At least $2,000 must remain in cash. [P2]"

## 8. Generate the answer

In [ ]:
answer = generate_answer(question, context)
print(answer)

## 9. Deliberate limitation: vague query

In [ ]:
vague = retrieve_chunks("What about it?", chunks, vectorizer, matrix)
for chunk, score in vague:
    print(chunk["id"], round(float(score), 3))

## 10. Rewrite the query

In [ ]:
rewritten = "What is the minimum cash requirement?"
better = retrieve_chunks(rewritten, chunks, vectorizer, matrix)
for chunk, score in better:
    print(chunk["id"], round(float(score), 3))

## 11. Complete result

In [ ]:
better_context = build_context(better)
print(generate_answer(rewritten, better_context))

## 12. Concise LlamaIndex equivalent (optional)

In [ ]:
try:
    from llama_index.core import Document, VectorStoreIndex
except ImportError:
    print("Optional LlamaIndex example skipped: install llama-index-core to run it.")
else:
    if os.getenv("COURSEWARE_OFFLINE", "0") == "1":
        print("Optional LlamaIndex example skipped in offline mode.")
    else:
        index = VectorStoreIndex.from_documents([Document(text=policy_text)])
        print(index.as_query_engine().query(question))

## Takeaways
- RAG is load, split, represent, retrieve, and answer.
- Visible scores make retrieval limitations inspectable.
- Grounded prompts carry both context and citation IDs.